# Data Analysis of the filtered Materials dataset

---

In [1]:
import duckdb

from src.data_preprocessing.config import INPUT_FILE_MATERIALS_2w_b

mat_data = INPUT_FILE_MATERIALS_2w_b

In [2]:
con = duckdb.connect()

In [3]:
# Attach the parquet file as a table
con.execute(f"CREATE OR REPLACE TABLE materials AS SELECT * FROM '{mat_data}';")

### Missing percentage per column

In [4]:
print("Null percentage per column:")

# Get list of all column names using DESCRIBE
columns = con.execute(f"DESCRIBE SELECT * FROM '{mat_data}'").fetchdf()['column_name'].tolist()

results = []
total_rows = con.execute(f"SELECT COUNT(*) FROM '{mat_data}'").fetchone()[0]

for col in columns:
    null_count = con.execute(f"SELECT COUNT(*) - COUNT({col}) FROM '{mat_data}'").fetchone()[0]
    null_percentage = round(100.0 * null_count / total_rows, 2)
    results.append((col, null_percentage))

# Sort and print
results.sort(key=lambda x: x[1], reverse=True)
for col, perc in results:
    print(f"{col}: {perc:.2f}% nulls")

Null percentage per column:
lot_packed_at: 100.00% nulls
supplier_order_desc: 100.00% nulls
mounting_place: 2.33% nulls
supplier_order_date_code: 1.24% nulls
setup_started_at: 0.00% nulls
component_position: 0.00% nulls
component_id: 0.00% nulls
component_index_id: 0.00% nulls
supplier_order_id: 0.00% nulls
created_at: 0.00% nulls
workstep_number_mes: 0.00% nulls
station_id: 0.00% nulls
station_number: 0.00% nulls
station_desc: 0.00% nulls
sequence_number: 0.00% nulls
book_state: 0.00% nulls
serial_number_id: 0.00% nulls
serial_number: 0.00% nulls
workorder_id: 0.00% nulls
workorder_number: 0.00% nulls
workorder_desc: 0.00% nulls
part_number: 0.00% nulls
part_desc: 0.00% nulls
panel_position: 0.00% nulls
supplier_id: 0.00% nulls
supplier_code: 0.00% nulls
supplier_name: 0.00% nulls
container_number: 0.00% nulls
supplier_order_number: 0.00% nulls


### Unique value count per column (Cardinality)

In [5]:
# Get column names
columns = con.execute("SELECT name FROM pragma_table_info('materials');").fetchdf()['name'].tolist()

# Calculate unique counts per column
results = []
for col in columns:
    unique_count = con.execute(f"SELECT COUNT(DISTINCT {col}) FROM materials;").fetchone()[0]
    results.append((col, unique_count))

# Print nicely
import pandas as pd

unique_counts = pd.DataFrame(results, columns=['column_name', 'unique_values'])
print("\n=== Unique Values per Column ===")
print(unique_counts)

categorical_cols = con.execute("""
                               SELECT name
                               FROM pragma_table_info('materials')
                               WHERE type IN ('VARCHAR', 'STRING', 'TEXT');
                               """).fetchdf()['name'].tolist()

for col in categorical_cols:
    print(f"\n=== Top 10 Frequent Values for '{col}' ===")
    result = con.execute(f"""
        SELECT {col} AS value, COUNT(*) AS freq
        FROM materials
        GROUP BY {col}
        ORDER BY freq DESC
        LIMIT 10;
    """).fetchdf()
    print(result)


=== Unique Values per Column ===
                 column_name  unique_values
0           setup_started_at           1943
1         component_position            249
2               component_id            166
3         component_index_id            364
4          supplier_order_id           2679
5                 created_at            418
6        workstep_number_mes              3
7                 station_id              3
8             station_number              3
9               station_desc              3
10           sequence_number              3
11                book_state              3
12          serial_number_id            188
13             serial_number            188
14              workorder_id             17
15          workorder_number             17
16            workorder_desc              4
17               part_number              4
18                 part_desc              3
19             lot_packed_at              0
20            panel_position              

### Dataset summary: number of rows, memory usage (approximate)

In [ ]:
# Get total number of rows
summary = con.execute("""
                      SELECT COUNT(*) AS total_rows
                      FROM materials;
                      """).fetchdf()

print("\n=== Dataset Summary ===")
print(summary)

# Approximate size: Sum LENGTH for string columns
string_cols = con.execute("""
                          SELECT name
                          FROM pragma_table_info('materials')
                          WHERE type IN ('VARCHAR', 'STRING', 'TEXT');
                          """).fetchdf()['name'].tolist()

if string_cols:
    length_sum_expr = " + ".join([f"LENGTH({col})" for col in string_cols])
    size_query = f"""
        SELECT ROUND(SUM({length_sum_expr}) / 1024 / 1024, 2) AS approx_string_data_MB
        FROM materials;
    """
    approx_size = con.execute(size_query).fetchdf()
    print("\n=== Approximate Size of String Data (MB) ===")
    print(approx_size)
else:
    print("\nNo string columns to estimate size.")

# Schema info
schema = con.execute("""
                     SELECT name AS column_name, type AS data_type
                     FROM pragma_table_info('materials');
                     """).fetchdf()
print("\n=== Schema Information ===")
print(schema)

In [ ]:
con.close()